# Guest Service Resolution Time Prediction

**Revised Task 3 experiment.** This regression objective replaces the original Concierge Request Router assignment. The available records lack guest-message text, validated urgency/department labels, and confirmed SLA information for routing.

> The experiment predicts the recorded `Service_Resolution_Time_Min` column. Available files do not establish that every target is an observed elapsed time from actual submission through resolution. These are exploratory results, not an operationally validated or deployable model.


## 1. Project overview and revised objective

Estimate how long a guest service request may take to resolve using information available when it is submitted, before hotel staff begin processing. A good estimate could help set guest expectations and plan capacity. This notebook tests whether available predictors improve over a constant mean estimate. It displays aggregate findings only and creates no model artifact or API.

## 2. Problem definition and regression approach

Target: `Service_Resolution_Time_Min`. Compare a mean `DummyRegressor`, Linear Regression, and Random Forest Regressor. MAE/RMSE are in minutes; R² compares squared error against the test-set target-mean reference.

## 3. Dataset description and source relationships

`Service_Requests.csv` and `Bookings.csv` each had 500 rows and joined one-to-one on `Booking_ID`: all 500 keys matched, with no loss or duplication. The merged CSV has 500 rows and 27 columns. The cleaned workbook also has 500 payment rows and 431 guest-profile rows. Guest profiles have repeated IDs and conflicting values, so the Guests sheet is not joined; payments add no verified service workflow features.

### Input paths

By default, place these files in repository-relative `data/`:

- `data/Service_Requests_Bookings_Merged.csv`
- `data/Task3_Baseline_Split_Manifest.csv`

The notebook does not bundle, copy, or synthesize row-level data. For local validation only, set `TASK3_DATA_DIR` to the existing Task 3 working folder. The loading cell gives a helpful error listing missing inputs.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

cwd=Path.cwd()
REPO_ROOT=cwd.parent if cwd.name=='notebooks' else cwd
DATA_DIR=Path(os.environ.get('TASK3_DATA_DIR',str(REPO_ROOT/'data')))
MERGED_PATH=DATA_DIR/'Service_Requests_Bookings_Merged.csv'
SPLIT_PATH=DATA_DIR/'Task3_Baseline_Split_Manifest.csv'
missing=[p for p in (MERGED_PATH,SPLIT_PATH) if not p.is_file()]
if missing:
    expected='\n'.join(f'- {p}' for p in missing)
    raise FileNotFoundError('Task 3 inputs are missing. Put both files in repository data/ by default, or set TASK3_DATA_DIR for local validation.\n'+expected)
source=pd.read_csv(MERGED_PATH)
manifest=pd.read_csv(SPLIT_PATH)
print(f'Loaded merged source: {len(source)} records, {source.shape[1]} columns; split assignments: {len(manifest)}.')
print('No guest-level rows are printed or saved by this notebook.')


## 4. Data assessment, quality checks, and integrity findings

The merged source repeats target and selected request fields from both input tables. Earlier key-level validation found the source and cleaned versions consistent. This cell repeats aggregate checks without displaying identifiers or individual rows.


In [ ]:
required={'Booking_ID','Service_Request__Request_ID','Service_Request__Service_Request_Type','Service_Request__Service_Resolution_Time_Min','Service_Request__Request_Date','Booking__Check_In','Booking__Room_Type','Booking__Guest_Segment','Booking__Booking_Channel','Booking__Booking_Status','Booking__Service_Resolution_Time_Min'}
missing_cols=sorted(required-set(source.columns))
if missing_cols: raise ValueError(f'Missing required columns: {missing_cols}')
y=pd.to_numeric(source['Service_Request__Service_Resolution_Time_Min'],errors='raise')
y_booking=pd.to_numeric(source['Booking__Service_Resolution_Time_Min'],errors='raise')
request_date=pd.to_datetime(source['Service_Request__Request_Date'],format='%Y-%m-%d',errors='raise')
checkin=pd.to_datetime(source['Booking__Check_In'],format='%Y-%m-%d',errors='raise')
print(f'Rows={len(source)}; missing Booking_ID={int(source.Booking_ID.isna().sum())}; duplicate Booking_ID rows={int(source.Booking_ID.duplicated().sum())}; duplicate Request_ID rows={int(source["Service_Request__Request_ID"].duplicated().sum())}.')
print(f'Request/booking target-copy mismatches={int(y.ne(y_booking).sum())}; Request_Date equals Check_In={int(request_date.dt.date.eq(checkin.dt.date).sum())}/{len(source)}.')
print(f'Request_Date is date-only YYYY-MM-DD for {int(source["Service_Request__Request_Date"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}").sum())}/{len(source)} rows.')


## 5. Target variable analysis: `Service_Resolution_Time_Min`

The recorded target is present for all 500 rows and ranges from 14 to 135 minutes. This describes the column; it does not prove its measurement method.


In [ ]:
print(pd.DataFrame({'n':[int(y.count())],'minimum_minutes':[y.min()],'median_minutes':[y.median()],'mean_minutes':[y.mean()],'maximum_minutes':[y.max()]}).to_string(index=False,float_format=lambda v:f'{v:.2f}'))


## 6. Cancelled and Abandoned booking records

There are 64 Cancelled and 57 Abandoned booking statuses. All 121 have target values, but source files have no ticket completion state or event history proving service was completed. Dataset A keeps all rows; Dataset B is a sensitivity comparison, not a claim that these targets are invalid.


In [ ]:
status=source['Booking__Booking_Status'].astype(str)
type_col='Service_Request__Service_Request_Type'
by_status=source.assign(_status=status,_target=y).groupby('_status')['_target'].agg(records='count',minimum='min',median='median',mean='mean',maximum='max').reindex(['Completed','Cancelled','Abandoned'])
print('Recorded target by booking status (minutes):')
print(by_status.to_string(float_format=lambda v:f'{v:.2f}'))
print('Request type counts by status:')
print(pd.crosstab(status,source[type_col]).reindex(['Completed','Cancelled','Abandoned']).to_string())


## 7. Feature selection and leakage prevention

Six provisional inputs: `Service_Request_Type`, `Room_Type`, `Guest_Segment`, `Booking_Channel`, `Request_Month`, and `Request_DayOfWeek`. The category and booking fields plausibly exist at intake, but point-in-time field histories are absent. `Request_Date` is date-only and equals Check_In in every row, so calendar timing is uncertain.

Excluded: `Service_Response_Time_Min` and `Guest_Satisfaction_1_5` (post-submission); all identifiers; `Booking_Status` (audit-only final status); the duplicated booking target; conflicting guest-profile records; and financial/whole-period history fields with uncertain timing or leakage risk.

## 8. Preparation of Dataset A and Dataset B

Dataset A keeps all 500 rows. Dataset B keeps `Booking_Status == Completed` (379 rows). Month and weekday are derived from Request_Date. The prepared frames contain six predictors, the request-side target, and audit-only status; status and identifiers are never passed to estimators.


In [ ]:
TARGET='Service_Resolution_Time_Min'
FEATURES=['Service_Request_Type','Room_Type','Guest_Segment','Booking_Channel','Request_Month','Request_DayOfWeek']
CAT_FEATURES=['Service_Request_Type','Room_Type','Guest_Segment','Booking_Channel','Request_DayOfWeek']
NUM_FEATURES=['Request_Month']
base=pd.DataFrame({'Service_Request_Type':source['Service_Request__Service_Request_Type'],'Room_Type':source['Booking__Room_Type'],'Guest_Segment':source['Booking__Guest_Segment'],'Booking_Channel':source['Booking__Booking_Channel'],'Request_Month':request_date.dt.month.astype(int),'Request_DayOfWeek':request_date.dt.day_name(),TARGET:y.astype(float),'Booking_Status':status})
datasets={'A':base.reset_index(drop=True),'B':base.loc[status.eq('Completed')].reset_index(drop=True)}
for key,frame in datasets.items(): print(f'Dataset {key}: {len(frame)} rows; missing predictor/target values={int(frame[FEATURES+[TARGET]].isna().sum().sum())}.')


## 9. Train/test split and reproducibility

The saved `Task3_Baseline_Split_Manifest.csv` preserves the original seed-42 Fisher–Yates assignments. It is used directly, rather than silently replaced by a new random split or scikit-learn `train_test_split`. Dataset B indices refer to the order-preserving Completed subset.


In [ ]:
if set(manifest.columns)!={'Dataset','Row_Index','Split'}: raise ValueError('Unexpected split-manifest schema.')
splits={}
for key,frame in datasets.items():
    part=manifest.loc[manifest.Dataset.astype(str).eq(key)].copy(); part['Row_Index']=pd.to_numeric(part.Row_Index,errors='raise').astype(int)
    train_idx=part.loc[part.Split.eq('train'),'Row_Index'].to_numpy(); test_idx=part.loc[part.Split.eq('test'),'Row_Index'].to_numpy()
    ntrain,ntest=(400,100) if key=='A' else (303,76)
    if len(frame)!=ntrain+ntest or len(train_idx)!=ntrain or len(test_idx)!=ntest: raise ValueError(f'Dataset {key}: split size mismatch.')
    if set(train_idx)&set(test_idx) or set(train_idx)|set(test_idx)!=set(range(len(frame))): raise ValueError(f'Dataset {key}: split does not partition all rows.')
    splits[key]=(train_idx,test_idx); print(f'Dataset {key}: train={len(train_idx)}, test={len(test_idx)}; saved manifest applied.')


## 10. Mean-prediction baseline

`DummyRegressor(strategy='mean')` learns the training-set target mean and predicts that constant for every test row. This is a no-feature benchmark. A feature model must improve on it to show the available inputs add predictive value.

## 11. Linear Regression and Random Forest experiments

Both models use a pipeline: one-hot encoding for the five categorical fields with `handle_unknown='ignore'`, pass-through numeric Request_Month, preprocessing fitted on training rows only. Random Forest uses `random_state=42` and default 100 trees. No tuning is performed.


In [ ]:
def make_pipeline(estimator):
    prep=ColumnTransformer([('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),CAT_FEATURES),('numeric','passthrough',NUM_FEATURES)],remainder='drop')
    return Pipeline([('preprocess',prep),('regressor',estimator)])
def scores(actual,pred): return {'MAE_Minutes':float(mean_absolute_error(actual,pred)),'RMSE_Minutes':float(np.sqrt(mean_squared_error(actual,pred))),'R2':float(r2_score(actual,pred))}
models={'DummyRegressor':lambda:DummyRegressor(strategy='mean'),'Linear Regression':lambda:make_pipeline(LinearRegression()),'Random Forest':lambda:make_pipeline(RandomForestRegressor(random_state=42,n_jobs=1))}


## 12. Evaluation using MAE, RMSE, and R²

Each estimator uses the same saved train/test rows per dataset. Baseline metrics are checked to four decimal places against the prior verified assessment.


In [ ]:
result_rows=[]; predictions={}
for key,frame in datasets.items():
    tr,te=splits[key]; X=frame[FEATURES]; target=frame[TARGET].to_numpy(dtype=float); Xtr,Xte=X.iloc[tr],X.iloc[te]; ytr,yte=target[tr],target[te]
    for name,factory in models.items():
        model=factory(); model.fit(Xtr,ytr); pred=np.asarray(model.predict(Xte),dtype=float)
        result_rows.append({'Dataset':key,'Model':name,'Train_Rows':len(tr),'Test_Rows':len(te),**scores(yte,pred)})
        if name!='DummyRegressor': predictions[(key,name)]=(yte.copy(),pred.copy(),frame.iloc[te]['Service_Request_Type'].to_numpy())
results=pd.DataFrame(result_rows)
expected={'A':(25.1491,28.6496),'B':(25.2302,30.5012)}
for key,(mae,rmse) in expected.items():
    row=results.loc[(results.Dataset==key)&(results.Model=='DummyRegressor')].iloc[0]
    if not np.isclose(row.MAE_Minutes,mae,atol=.00005) or not np.isclose(row.RMSE_Minutes,rmse,atol=.00005): raise AssertionError(f'Dataset {key} baseline differs from verified result.')
print(results.to_string(index=False,float_format=lambda v:f'{v:.4f}'))
print('Verified baseline MAE/RMSE reproduced to four decimal places.')


## 13. Cross-validation and model diagnostics

Five-fold shuffled CV (seed 42) uses only the training partitions, preserving the saved test sets. Below, eta-squared summarizes training-only differences between category means (descriptive, not causal); residual summaries are aggregate and omit identifiers.


In [ ]:
folds=KFold(n_splits=5,shuffle=True,random_state=42); cv_rows=[]
for key,frame in datasets.items():
    tr,_=splits[key]; Xtr=frame.iloc[tr][FEATURES]; ytr=frame.iloc[tr][TARGET].to_numpy(dtype=float)
    for name,factory in models.items():
        cvout=cross_validate(factory(),Xtr,ytr,cv=folds,scoring={'mae':'neg_mean_absolute_error','rmse':'neg_root_mean_squared_error'},return_train_score=False,n_jobs=1)
        maef=-cvout['test_mae']; rmsef=-cvout['test_rmse']; cv_rows.append({'Dataset':key,'Model':name,'MAE_Mean':maef.mean(),'MAE_SD':maef.std(ddof=1),'RMSE_Mean':rmsef.mean(),'RMSE_SD':rmsef.std(ddof=1)})
cv_results=pd.DataFrame(cv_rows); print('Five-fold CV on training rows:'); print(cv_results.to_string(index=False,float_format=lambda v:f'{v:.2f}'))
assoc=[]
for key,frame in datasets.items():
    tr=frame.iloc[splits[key][0]]; yy=tr[TARGET].astype(float); total=float(((yy-yy.mean())**2).sum())
    for feat in FEATURES:
        between=sum(len(v)*(float(v.mean())-float(yy.mean()))**2 for _,v in tr.groupby(feat,observed=True)[TARGET]); assoc.append({'Dataset':key,'Feature':feat,'Eta_Squared':between/total if total else 0})
print('Category mean separation eta-squared, training only:'); print(pd.DataFrame(assoc).pivot(index='Feature',columns='Dataset',values='Eta_Squared').to_string(float_format=lambda v:f'{v:.3f}'))
residual_rows=[]; type_rows=[]
for (key,name),(actual,pred,types) in predictions.items():
    residual=actual-pred; residual_rows.append({'Dataset':key,'Model':name,'Mean_Signed_Residual_Min':residual.mean(),'Underpredicted':int((residual>0).sum()),'Overpredicted':int((residual<0).sum())})
    for typ in sorted(set(types)):
        mask=types==typ; type_rows.append({'Dataset':key,'Model':name,'Request_Type':typ,'Test_Count':int(mask.sum()),'MAE_Minutes':float(np.abs(residual[mask]).mean())})
print('Residual direction (positive signed residual means underprediction):'); print(pd.DataFrame(residual_rows).to_string(index=False,float_format=lambda v:f'{v:.2f}'))
print('Aggregate test MAE by request type:'); print(pd.DataFrame(type_rows).to_string(index=False,float_format=lambda v:f'{v:.2f}'))


## 14. Comparison of results

**Held-out metrics from the verified experiment** (minutes except R²):

| Dataset | Model | MAE | RMSE | R² |
|---|---|---:|---:|---:|
| A (500) | DummyRegressor | 25.1491 | 28.6496 | -0.0054 |
| A (500) | Linear Regression | 25.5807 | 29.5887 | -0.0724 |
| A (500) | Random Forest | 26.0896 | 29.7407 | -0.0835 |
| B (379 Completed) | DummyRegressor | 25.2302 | 30.5012 | -0.0193 |
| B (379 Completed) | Linear Regression | 26.1583 | 31.5220 | -0.0886 |
| B (379 Completed) | Random Forest | 26.6044 | 32.1884 | -0.1352 |

**Five-fold CV on the training data** (mean ± sample SD, minutes):

| Dataset | Model | MAE | RMSE |
|---|---|---:|---:|
| A | DummyRegressor | 23.32 ± 1.37 | 27.61 ± 1.42 |
| A | Linear Regression | 23.95 ± 1.61 | 28.57 ± 1.68 |
| A | Random Forest | 23.88 ± 1.68 | 28.81 ± 1.73 |
| B | DummyRegressor | 23.43 ± 1.03 | 27.53 ± 1.45 |
| B | Linear Regression | 24.84 ± 0.47 | 29.75 ± 1.18 |
| B | Random Forest | 24.55 ± 0.57 | 29.19 ± 1.23 |

The baseline wins on held-out MAE/RMSE/R² for both datasets, and CV favors it on mean MAE/RMSE. An MAE near 25 minutes means an average absolute difference of about 25 minutes from the recorded target. Negative R² means squared test error exceeds the test-target-mean reference; that reference is unavailable at prediction time. A and B test membership differs, so score differences cannot be attributed only to filtering.


## 15. Limitations, interpretation, and recommendations

- Target copies agree across files, but target clock definition and event provenance are absent.
- The 121 Cancelled/Abandoned targets remain unverified; plausible distributions do not prove completed service.
- Request_Date is date-only and equals Check_In in all 500 rows; actual request hour is unavailable.
- Conflicting Guest_ID profiles prevent a safe Guests join.
- Request complexity, priority, department, staffing, queue/backlog, concurrent workload, and ticket lifecycle data are absent.
- Evaluation is one saved holdout plus five-fold training-only CV on a small dataset; no temporal, external, prospective, or operational validation exists.

Before a stronger experiment, verify the target definition and 121 records against ticket logs; obtain actual request/resolution timestamps; reconcile guest identity and point-in-time feature definitions; and capture intake detail and operating capacity. Preserve a mean baseline and a future/time-based holdout. Do not deploy on these results.

## 16. Final conclusion

The initial experiments did **not** establish a useful predictive model. DummyRegressor outperformed Linear Regression and Random Forest on both datasets and cross-validation supported the same pattern. These results apply only to the recorded target and provisional inputs. The data is not validated for operational prediction, and no deployable model or prediction API is produced.
